In [1]:
!pip install kaggle
!mkdir -p ~/.kaggle
upload_file_path = '/content/kaggle.json'
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
The syntax of the command is incorrect.
'cp' is not recognized as an internal or external command,
operable program or batch file.
'chmod' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
from huggingface_hub import login
from google.colab import userdata

secret_label = "HFHub"
secret_value = userdata.get(secret_label)
login(token=secret_value)

D:\private\Coding\Aliaa Graduation Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
%pip install \
    datasets \
    evaluate \
    rouge_score\
    loralib \
    evaluate \
    accelerate \
    bitsandbytes \
    trl \
    peft \
    -U --quiet

In [ ]:
# Note: Remove this cell if using datasets>=4.4.1 which requires pyarrow>=21.0.0
# The pyarrow downgrade causes compatibility issues
# If you need pyarrow 19.0.0 for other reasons, pin datasets version:
# !pip install datasets==3.0.0 pyarrow==19.0.0

# For most cases, just use the latest compatible versions:
# !pip install pyarrow>=21.0.0

In [ ]:
!pip install tf-keras

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import pandas as pd
import re
import numpy as np
import string
from nltk.corpus import stopwords
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfTransformer,TfidfVectorizer
from sklearn.pipeline import Pipeline
import evaluate

In [ ]:
# !pip install -q -U git+https://github.com/huggingface/peft.git

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

In [ ]:
import transformers

torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    quantization_config=quant_config,
    device_map={"": 0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:
# Set pad_token as end-of-sentence token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(model))

## <b>3 <span style='color:#9146ff'>|</span> Data Preparation </b>


In [ ]:
import os
import zipfile
import glob

# Check if data already exists in content folder or current directory
# The parquet file has a pattern like: train-00000-of-00001-*.parquet
parquet_patterns = [
    'content/train-*.parquet',
    'train-*.parquet',
    '/content/train-*.parquet'
]

existing_file = None
for pattern in parquet_patterns:
    matches = glob.glob(pattern)
    if matches:
        existing_file = matches[0]
        break

if existing_file:
    print(f"✓ Found dataset locally: {existing_file} - skipping download")
else:
    print(f"Downloading Arabic Instruct Chatbot dataset...")
    # Download dataset
    !kaggle datasets download -d omgits0mar/arabic-instruct-chatbot-dataset
    
    # Cross-platform unzip (works on Windows and Linux)
    zip_file = 'arabic-instruct-chatbot-dataset.zip'
    if os.path.exists(zip_file):
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✓ Extracted {zip_file} successfully")
        # Clean up zip file
        os.remove(zip_file)
        # Find the extracted file
        matches = glob.glob('train-*.parquet')
        if matches:
            print(f"✓ Dataset ready: {matches[0]}")
    else:
        print(f"Error: {zip_file} not found after download")

In [ ]:
import pandas as pd
import glob

# Find the parquet file dynamically
parquet_patterns = [
    'content/train-*.parquet',
    'train-*.parquet',
    '/content/train-*.parquet'
]

parquet_file = None
for pattern in parquet_patterns:
    matches = glob.glob(pattern)
    if matches:
        parquet_file = matches[0]
        break

if parquet_file:
    print(f"Loading dataset from: {parquet_file}")
    df = pd.read_parquet(parquet_file)
    print(f"✓ Loaded {len(df)} rows")
    df.head()
else:
    raise FileNotFoundError("Arabic chatbot dataset not found. Please run the download cell first.")

## <b>4 <span style='color:#9146ff'>|</span> Data Preprocessing </b>


In [ ]:
# Calculate the maximum length of text in the 'instruction' column
max_length_instruction = df['instruction'].apply(len).mean()

# Calculate the maximum length of text in the 'output' column
max_length_output = df['output'].apply(len).mean()

# Print the results
print(f"Maximum length of 'instruction': {max_length_instruction}")
print(f"Maximum length of 'output': {max_length_output}")

In [ ]:
def tokenize_function(row):
    # Tokenize the conversations
    question = ' '.join(row["instruction"]) if isinstance(row["instruction"], list) else row["instruction"]

    row['input_ids'] = tokenizer(question, padding="max_length", truncation=True, max_length = 128, return_tensors="pt").input_ids[0]

    # Assuming "answer" column is already a string, no need for conversion
    row['labels'] = tokenizer(row["output"], padding="max_length", truncation=True, max_length = 256, return_tensors="pt").input_ids[0]

    return row


# Tokenize the DataFrame
tokenized_df = df.apply(tokenize_function, axis=1)

In [ ]:
# Convert columns to list
tokenized_df['input_ids'] = tokenized_df['input_ids'].apply(lambda x: x.tolist())
tokenized_df['labels'] = tokenized_df['labels'].apply(lambda x: x.tolist())

In [ ]:
tokenized_df

In [ ]:
from datasets import Dataset

# Assuming `tokenized_df` is your pandas DataFrame
dataset = Dataset.from_pandas(tokenized_df[:10000])

In [ ]:
dataset

In [ ]:
from datasets import Dataset

# Use full dataset for better training (or subset for faster experimentation)
# Using 10000 samples as in original, but you can increase this
dataset_full = Dataset.from_pandas(tokenized_df[:10000])

# Combine 'instruction' and 'output' into a 'text' column for SFTTrainer
dataset_full = dataset_full.map(lambda x: {"text": x["instruction"] + " " + x["output"]}, batched=False)

# Split into train and test sets (90% train, 10% test)
train_test_split = dataset_full.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
#Model Training and Fine-tuning

In [ ]:
# Load LoRA configuration
peft_args = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# Set training parameters
training_params = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    eval_strategy="steps",  # Enable evaluation during training
    eval_steps=200,  # Evaluate every 200 steps
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    load_best_model_at_end=True,  # Load best model after training
    metric_for_best_model="eval_loss",
)

In [ ]:
from peft import get_peft_model, TaskType

peft_model = get_peft_model(model,
                            peft_args)
print(print_number_of_trainable_model_parameters(peft_model))

In [ ]:
from trl import SFTTrainer
help(SFTTrainer)


### SFTTrainer:

- Supervised Fine-tuning (SFT): Optimized for fine-tuning pre-trained models with smaller datasets on supervised learning tasks.
- Simpler interface: Provides a streamlined workflow with fewer configuration options, making it easier to get started.
- Efficient memory usage: Uses techniques like parameter-efficient (PEFT) and packing optimizations to reduce memory consumption during training.
- Faster training: Achieves comparable or better accuracy with smaller datasets and shorter training times than Trainer.

In [ ]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_args,
    dataset_text_field="text",
    max_seq_length=512,  # Increased for Arabic text which can be longer
    tokenizer=tokenizer,
    args=training_params,
    packing=False,
)

### Training

In [ ]:
trainer.train()

### Save model & Publish

In [ ]:
trainer.model.save_pretrained("./llama-3-8B-Arabic")
tokenizer.save_pretrained("./llama-3-8B-Arabic")

## Export LoRA Adapter for ML API

Copy the trained LoRA adapter and tokenizer to `ml_api/saved_models/chatbot/` so the FastAPI backend can load them.

In [ ]:
import os
import shutil
from pathlib import Path

# ============================================================
# Export LoRA adapter to ml_api/saved_models/chatbot/
# ============================================================
adapter_source = Path("./llama-3-8B-Arabic")
output_dir = Path("ml_api/saved_models/chatbot/llama-3-8B-Arabic")

if not adapter_source.exists():
    raise FileNotFoundError(
        f"LoRA adapter not found at {adapter_source}. "
        "Run the training cell (trainer.train()) and save cell first."
    )

# Create output directory and copy adapter files
output_dir.mkdir(parents=True, exist_ok=True)

# Copy all adapter files
copied_files = []
for f in adapter_source.iterdir():
    dst = output_dir / f.name
    if f.is_file():
        shutil.copy2(str(f), str(dst))
        size_mb = f.stat().st_size / 1e6
        copied_files.append((f.name, size_mb))
    elif f.is_dir():
        if dst.exists():
            shutil.rmtree(str(dst))
        shutil.copytree(str(f), str(dst))
        copied_files.append((f.name + "/", 0))

print("Exported LoRA adapter to ml_api/saved_models/chatbot/llama-3-8B-Arabic/")
print("=" * 60)
for fname, size in sorted(copied_files):
    if size > 0:
        print(f"  {fname:40s} {size:>8.2f} MB")
    else:
        print(f"  {fname:40s} (directory)")

# ============================================================
# Validation: check required files exist
# ============================================================
print("\n" + "=" * 60)
print("Validating exported adapter...")

required_files = [
    "adapter_config.json",       # LoRA config (rank, alpha, target modules)
    "adapter_model.safetensors", # LoRA weights (or adapter_model.bin)
    "tokenizer_config.json",     # Tokenizer config
    "tokenizer.json",            # Tokenizer vocabulary
    "special_tokens_map.json",   # Special tokens
]

errors = []
for fname in required_files:
    fpath = output_dir / fname
    # adapter_model could be .safetensors or .bin
    if fname == "adapter_model.safetensors" and not fpath.exists():
        alt = output_dir / "adapter_model.bin"
        if alt.exists():
            print(f"  OK: adapter_model.bin ({alt.stat().st_size / 1e6:.2f} MB)")
            continue
        else:
            errors.append(f"MISSING: adapter_model.safetensors (or .bin)")
            continue
    if fpath.exists():
        print(f"  OK: {fname} ({fpath.stat().st_size / 1e6:.2f} MB)")
    else:
        errors.append(f"MISSING: {fname}")

print("=" * 60)
if errors:
    print(f"\nERRORS ({len(errors)}):")
    for e in errors:
        print(f"  {e}")
else:
    print(f"\nAll adapter files validated successfully!")
    print(f"ML API will load adapter from: {output_dir.resolve()}")
    print(f"Base model: meta-llama/Meta-Llama-3-8B-Instruct (downloaded at runtime)")

In [ ]:
# model.push_to_hub("")
# tokenizer.push_to_hub("")

## <b>6 <span style='color:#9146ff'>|</span> Testing the model performance on a single inference </b>


In [ ]:
def single_inference(question):
    messages = [
        {"role": "system", "content": "اجب علي الاتي بالعربي فقط."},
    ]

    messages.append({"role": "user", "content": question})


    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.4,
    #     top_p=0.9,
    )
    response = outputs[0][input_ids.shape[-1]:]
    output = tokenizer.decode(response, skip_special_tokens=True)
    return output

In [ ]:
question = """ما هي طريقة عمل البيتزا , اجب في خطوات"""

answer = single_inference(question)

print(f'INPUT QUESTION:\n{question}')
print(f'\n\nModel Answer:\n{answer}')

In [ ]:
question = """   """

answer = single_inference(question)

print(f'INPUT QUESTION:\n{question}')
print(f'\n\nModel Answer:\n{answer}')